# DeepGuard — FaceForensics++ C23 Dataset Exploration

This notebook explores the FaceForensics++ C23 dataset, a benchmark dataset for deepfake detection. We will perform the following steps:

1.  **Install and Verify Environment**: Set up the necessary libraries and check the environment.
2.  **Load Dataset**: Load the FaceForensics++ C23 dataset in streaming mode.
3.  **Inspect Data**: Examine the dataset structure and individual samples.
4.  **Extract Metadata**: Process video paths to extract labels (REAL/FAKE) and manipulation methods.
5.  **Analyze Dataset Distribution**: Visualize the distribution of real vs. fake videos and different deepfake manipulation methods.
6.  **Save Results**: Store the generated metadata and analysis summaries for future use.

## 1. Install Dependencies

This section installs the required Python libraries, including `pyarrow`, `datasets`, and `torchcodec`. Note that `pyarrow` is uninstalled and then reinstalled to ensure a specific compatible version.

In [ ]:
!pip uninstall -y pyarrow -q
!pip install -q --no-cache-dir pyarrow==21.0.0
!pip install -q datasets huggingface_hub torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 185.2 MB/s eta 0:00:00


## 2. Verify Environment

Here, we verify that the installed libraries and Python environment are correctly set up, including checking for GPU availability for PyTorch.

In [ ]:
import sys
import pyarrow
import datasets
import torch

print("Python       :", sys.version)
print("PyArrow      :", pyarrow.__version__)
print("Datasets     :", datasets.__version__)
print("PyTorch      :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

Python       : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyArrow      : 21.0.0
Datasets     : 4.0.0
PyTorch      : 2.11.0+cu128
CUDA available: True
GPU          : Tesla T4


## 3. Import Libraries

This section imports all necessary libraries for data processing, analysis, and visualization, such as `pandas`, `numpy`, `matplotlib`, and `datasets`.

In [ ]:
import os
import re
import json
import math
import tempfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

print("Libraries imported successfully.")

Libraries imported successfully.


## 4. Mount Google Drive and Define Project Structure

We mount Google Drive to store analysis results and define the project directory structure for organized output.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/DeepGuard")

ANALYSIS_DIR = PROJECT_DIR / "dataset_analysis"
GRAPH_DIR = ANALYSIS_DIR / "graphs"
SAMPLE_DIR = ANALYSIS_DIR / "sample_frames"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Analysis directory:", ANALYSIS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/DeepGuard
Analysis directory: /content/drive/MyDrive/DeepGuard/dataset_analysis


## 5. Dataset Configuration

This cell defines the ID of the dataset to be loaded from the Hugging Face Hub.

In [ ]:
DATASET_ID = "bitmind/FaceForensicsC23"

print("Dataset:", DATASET_ID)

Dataset: bitmind/FaceForensicsC23


## 6. Load Dataset in Streaming Mode

The FaceForensics++ C23 dataset is loaded in streaming mode, which is efficient for large datasets as it loads data on-the-fly rather than all at once. We load the 'train' split.

In [ ]:
ffpp = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True
)

print(ffpp)

Repo card metadata block was not found. Setting CardData to empty.


IterableDataset({
    features: ['video'],
    num_shards: 1
})


## 7. Inspect Dataset Features

We inspect the features of the loaded dataset to understand its structure, specifically the 'video' feature type.

In [ ]:
print("Dataset features:")
print(ffpp.features)

Dataset features:
{'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact')}


## 8. Inspect First Sample

This section examines the first sample from the dataset to understand the format of the video data. Initially, the video is decoded into a `VideoDecoder` object.

In [ ]:
first_sample = next(iter(ffpp))

print("Keys:")
print(first_sample.keys())

print("\nSample:")
for key, value in first_sample.items():
    print("\n ", key)
    print(type(value))
    print(value if not isinstance(value, bytes) else f"<bytes: {len(value)}> ")

Keys:
dict_keys(['video'])

Sample:

  video
<class 'torchcodec.decoders._video_decoder.VideoDecoder'>


## 9. Disable Video Decoding

To efficiently extract metadata without loading potentially large video frames, we disable video decoding. This allows us to inspect only the video's path and other metadata.

In [ ]:
ffpp_raw = ffpp.decode(False)

print("Video decoding disabled.")
print("This allows metadata/path inspection without decoding frames.")

Video decoding disabled.
This allows metadata/path inspection without decoding frames.


## 10. Inspect Raw Video Sample (Metadata Only)

After disabling decoding, we inspect a raw sample to see the metadata available, which primarily includes the video file path.

In [ ]:
raw_sample = next(iter(ffpp_raw))

print(raw_sample.keys())

for key, value in raw_sample.items():
    print("\nKEY:", key)
    print("TYPE:", type(value))

    if isinstance(value, dict):
        print(value.keys())

        for k, v in value.items():
            if isinstance(v, bytes):
                print(f"  {k}: <{len(v)} bytes>")
            else:
                print(f"  {k}: {v}")
    else:
        print(value)

dict_keys(['video'])

KEY: video
TYPE: <class 'dict'>
dict_keys(['bytes', 'path'])
  bytes: None
  path: zip://FaceForensics++_C23/fake/DeepFakeDetection/01_02__meeting_serious__YVGY8LOK.mp4::hf://datasets/bitmind/FaceForensicsC23@d8b51023ad41c6620ac70677d3145a1f418417ab/FaceForensics++_C23.zip


## 11. Define `parse_ffpp_path` Helper Function

This section defines the `parse_ffpp_path` function, crucial for extracting structured information (label, method, filename) from the complex video path strings within the dataset. It handles both 'real' and various 'fake' manipulation methods by parsing parts of the path.

In [ ]:
from pathlib import PurePosixPath

def parse_ffpp_path(path):

    if not path:
        return {
            "label": "UNKNOWN",
            "method": "UNKNOWN",
            "filename": None
        }

    # Get the part before ::hf://
    internal_path = path.split("::")[0]

    # Remove zip://
    internal_path = internal_path.replace("zip://", "")

    # Normalize
    internal_path = internal_path.replace("\\", "/")

    parts = PurePosixPath(internal_path).parts

    filename = PurePosixPath(internal_path).name

    label = "UNKNOWN"
    method = "UNKNOWN"

    # Expected structure:
    #
    # FaceForensics++_C23/
    #     fake/
    #         DeepFakeDetection/
    #             video.mp4
    #
    # OR
    #
    # FaceForensics++_C23/
    #     real/
    #         video.mp4

    lower_parts = [p.lower() for p in parts]

    if "real" in lower_parts:

        label = "REAL"
        method = "Real"

    elif "fake" in lower_parts:

        label = "FAKE"

        fake_methods = {
            "deepfakedetection": "DeepFakeDetection",
            "deepfakes": "Deepfakes",
            "face2face": "Face2Face",
            "faceshifter": "FaceShifter",
            "faceswap": "FaceSwap",
            "neuraltextures": "NeuralTextures"
        }

        for part in lower_parts:

            if part in fake_methods:

                method = fake_methods[part]
                break

    return {
        "label": label,
        "method": method,
        "filename": filename
    }

## 12. Stream ALL FF++ C23 Records (Full Dataset)

This section processes a larger number of videos (up to `MAX_VIDEOS`, here 7000) from the full dataset. It iterates through the streaming dataset, extracts video paths, and uses the `parse_ffpp_path` function to get detailed metadata (filename, label, method) for each video. The results are compiled into `full_metadata_df`.

In [ ]:
MAX_VIDEOS = 7000

records_full = []

stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True
)

stream = stream.decode(False)

for index, sample in enumerate(stream):

    if index >= MAX_VIDEOS:
        break

    video = sample.get("video")

    if not isinstance(video, dict):
        continue

    path = video.get("path")

    parsed = parse_ffpp_path(path)

    records_full.append({
        "index": index,
        "path": path,
        "filename": parsed["filename"],
        "label": parsed["label"],
        "method": parsed["method"]
    })

    if (index + 1) % 500 == 0:

        print(
            f"Processed {index + 1}/{MAX_VIDEOS}"
        )

full_metadata_df = pd.DataFrame(records_full)

print("\nFinished.")
print("Total records:", len(full_metadata_df))

Repo card metadata block was not found. Setting CardData to empty.


Processed 500/7000
Processed 1000/7000
Processed 1500/7000
Processed 2000/7000
Processed 2500/7000
Processed 3000/7000
Processed 3500/7000
Processed 4000/7000
Processed 4500/7000
Processed 5000/7000
Processed 5500/7000
Processed 6000/7000
Processed 6500/7000
Processed 7000/7000

Finished.
Total records: 7000


## 13. Full Dataset Statistics

This cell displays the overall statistics for the full dataset, including total video count, class distribution (real vs. fake), and the distribution of manipulation methods for fake videos.

In [ ]:
print("TOTAL VIDEOS")
print("=" * 40)

print(
    full_metadata_df["label"]
    .value_counts()
)

print("\nFAKE METHODS")
print("=" * 40)

print(
    full_metadata_df[
        full_metadata_df["label"] == "FAKE"
    ]["method"].value_counts()
)

TOTAL VIDEOS
label
FAKE    6000
REAL    1000
Name: count, dtype: int64

FAKE METHODS
method
DeepFakeDetection    1000
Deepfakes            1000
Face2Face            1000
FaceShifter          1000
FaceSwap             1000
NeuralTextures       1000
Name: count, dtype: int64


## 14. Save Full Metadata

The complete metadata DataFrame (`full_metadata_df`) is saved to a CSV file in the analysis directory. This file contains the detailed information for all processed videos.

In [ ]:
full_metadata_file = (
    ANALYSIS_DIR /
    "ffpp_c23_full_metadata.csv"
)

full_metadata_df.to_csv(
    full_metadata_file,
    index=False
)

print("Saved:")
print(full_metadata_file)

Saved:
/content/drive/MyDrive/DeepGuard/dataset_analysis/ffpp_c23_full_metadata.csv


## 15. Final Report & Summary

This cell generates a final, consolidated report summarizing the key findings of the dataset exploration, including dataset ID, total records, class distribution, and fake manipulation methods. It also confirms the location where analysis artifacts are saved.

In [ ]:
print("=" * 65)
print("DEEPGUARD — FACEFORENSICS++ C23 DATASET EXPLORATION")
print("=" * 65)

print("\nDataset:")
print(DATASET_ID)

print("\nTotal records:")
print(len(full_metadata_df))

print("\nClass distribution:")
print(
    full_metadata_df["label"]
    .value_counts()
)

print("\nFake manipulation methods:")
print(
    full_metadata_df[
        full_metadata_df["label"] == "FAKE"
    ]["method"].value_counts()
)

print("\nAnalysis directory:")
print(ANALYSIS_DIR)

print("\nCompleted successfully.")

DEEPGUARD — FACEFORENSICS++ C23 DATASET EXPLORATION

Dataset:
bitmind/FaceForensicsC23

Total records:
7000

Class distribution:
label
FAKE    6000
REAL    1000
Name: count, dtype: int64

Fake manipulation methods:
method
DeepFakeDetection    1000
Deepfakes            1000
Face2Face            1000
FaceShifter          1000
FaceSwap             1000
NeuralTextures       1000
Name: count, dtype: int64

Analysis directory:
/content/drive/MyDrive/DeepGuard/dataset_analysis

Completed successfully.


## 16. Advanced Visual Analysis — FaceForensics C23

This section focuses on loading the previously saved `ffpp_c23_full_metadata.csv` and conducting advanced visual analysis to gain deeper insights into the dataset's composition. It starts by loading the data and printing key statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load the complete metadata generated earlier
metadata_path = Path(
    "/content/drive/MyDrive/DeepGuard/dataset_analysis/ffpp_c23_full_metadata.csv"
)

df = pd.read_csv(metadata_path)

print("=" * 60)
print("FACEFORENSICS C23 — DATASET VISUAL ANALYSIS")
print("=" * 60)

print(f"Total videos : {len(df):,}")
print(f"Real videos  : {(df['label'] == 'REAL').sum():,}")
print(f"Fake videos  : {(df['label'] == 'FAKE').sum():,}")
print(f"Methods      : {df['method'].nunique()}")

FACEFORENSICS C23 — DATASET VISUAL ANALYSIS
Total videos : 7,000
Real videos  : 1,000
Fake videos  : 6,000
Methods      : 7


## 17. Real vs. Fake — Bar Chart

This bar chart visualizes the distribution of 'Real' and 'Fake' videos in the FaceForensicsC23 dataset, providing a clear comparison of class sizes with exact counts annotated on the bars.

In [ ]:
class_counts = df["label"].value_counts()

plt.figure(figsize=(9, 6))

bars = plt.bar(
    class_counts.index,
    class_counts.values,
    color=['#1f77b4', '#ff7f0e']
)

plt.title(
    "FaceForensicsC23 — Real vs Fake Videos",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Video Class", fontsize=12)
plt.ylabel("Number of Videos", fontsize=12)

for bar, value in zip(bars, class_counts.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 100,
        f"{value:,}",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

plt.ylim(0, max(class_counts.values) * 1.15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 900x600 with 1 Axes>

## 18. Manipulation Method Distribution

This bar chart illustrates the detailed distribution of manipulation methods across all fake videos in the dataset.

In [ ]:
method_counts = df[df["label"] == "FAKE"]["method"].value_counts()

plt.figure(figsize=(12, 6))
bars = plt.bar(
    method_counts.index,
    method_counts.values,
    color='#2ca02c'
)

plt.title(
    "FaceForensicsC23 — Fake Manipulation Methods Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Manipulation Method", fontsize=12)
plt.ylabel("Number of Videos", fontsize=12)
plt.xticks(rotation=30, ha="right")

for bar, value in zip(bars, method_counts.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 20,
        f"{value:,}",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

plt.ylim(0, max(method_counts.values) * 1.15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 1 Axes>